# Build 1 — Meeting Title Classifier: EDA
Exploratory analysis of the synthetic dataset before model training.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load data

In [ ]:
df = pd.read_csv('../data/synthetic/meeting_titles.csv')
print(f"Shape: {df.shape}")
df.head(10)

## 2. Label distribution

In [ ]:
counts = df['label'].value_counts()
print(counts.to_string())

fig, ax = plt.subplots()
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Meeting title label distribution')
for i, v in enumerate(counts):
    ax.text(v + 1, i, str(v), va='center')
plt.tight_layout()
plt.show()

## 3. Title length analysis

In [ ]:
df['char_len'] = df['title'].str.len()
df['word_count'] = df['title'].str.split().str.len()

print(df.groupby('label')[['char_len', 'word_count']].agg(['mean', 'min', 'max']).round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in sorted(df['label'].unique()):
    sub = df[df['label'] == label]
    axes[0].hist(sub['char_len'], bins=20, alpha=0.5, label=label)
    axes[1].hist(sub['word_count'], bins=12, alpha=0.5, label=label)

axes[0].set_title('Character length by category')
axes[0].set_xlabel('Characters')
axes[1].set_title('Word count by category')
axes[1].set_xlabel('Words')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Top tokens per category

In [ ]:
STOPWORDS = {'the', 'a', 'an', 'and', 'or', 'of', 'for', 'to', 'in', 'on', 'at', 'with', '-', '&', '/', '|', ':', '<>', 'x'}

def top_tokens(texts: pd.Series, n: int = 10) -> list[tuple[str, int]]:
    tokens = []
    for t in texts:
        for tok in re.split(r'[\s\-/|:&<>()\[\]]+', t.lower()):
            tok = tok.strip('.,!?"\'')
            if tok and tok not in STOPWORDS and len(tok) > 1:
                tokens.append(tok)
    return Counter(tokens).most_common(n)

for label in sorted(df['label'].unique()):
    top = top_tokens(df[df['label'] == label]['title'])
    print(f"\n{label.upper()}")
    print('  ' + ', '.join(f"{tok}({n})" for tok, n in top))

## 5. Signal words — do they leak across categories?

In [ ]:
# Check how many titles contain strong signal words
signals = {
    'standup':    ['standup', 'stand-up', 'dsu', 'daily', 'scrum'],
    'planning':   ['planning', 'roadmap', 'sprint', 'backlog', 'okr'],
    'one_on_one': ['1:1', '1-1', '1on1', '<>', '/'],
    'client':     list(map(str.lower, ['Acme', 'GlobalTech', 'RetailCo', 'HealthFirst', 'FinServ'])),
    'all_hands':  ['all hands', 'all-hands', 'town hall', 'allhands'],
    'interview':  ['interview', 'hiring', 'screen', 'onsite', 'debrief'],
    'workshop':   ['workshop', 'offsite', 'hackathon', 'deep dive', 'l&l'],
    'social':     ['lunch', 'happy hour', 'coffee', 'party', 'social'],
}

for label, words in signals.items():
    mask = df['title'].str.lower().apply(lambda t: any(w in t for w in words))
    by_label = df[mask]['label'].value_counts()
    leakage = {k: v for k, v in by_label.items() if k != label}
    if leakage:
        print(f"{label}: signal words also appear in → {leakage}")

## 6. Sample titles per category

In [ ]:
for label in sorted(df['label'].unique()):
    print(f"\n── {label.upper()} ──")
    for t in df[df['label'] == label]['title'].sample(8, random_state=42).tolist():
        print(f"  {t}")

## 7. Observations & notes for model training

Fill in after running the notebook:

- **Separability**: which categories look easiest / hardest to distinguish?
- **Ambiguous titles**: any titles that could belong to multiple categories?
- **Feature ideas**: signal tokens, length, special character patterns (`1:1`, `<>`, `[]`)
- **Data gaps**: any realistic title patterns missing from the generator?